# Metacatalog spectral modeling (Taylor expansion)

Post-hoc multi-point spectral fits for MHz-subband metacatalogs (15 bands, 18–82 MHz).

Each source with one or more valid `Total_flux_{band}` measurements is fit with a log-frequency **Taylor expansion** about ν₀ = **55 MHz**:

$$\ln S(\nu) = \sum_{j=0}^{n-1} a_j \left[\ln(\nu/\nu_0)\right]^j$$

Nested models with **1–4 terms** are compared. The selected model is the **simplest** whose reduced χ² is near the best fit; otherwise the lowest **BIC** wins.

**Defaults:** `Total_flux` (not peak), 15 MHz subbands, output columns prefixed `spec_`.

Set `USE_SYNTHETIC = True` to run against an inline demo catalog (no `CATALOG_DIR` required).

**Run cells in order.**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from lwa_catalog import CatalogLayout, read_metacatalog
from lwa_catalog.analyze import (
    SpectralFitConfig,
    fit_metacatalog_spectra,
    gather_band_flux_measurements,
    summarize_spectral_fit,
)
from lwa_catalog.analyze.spectral import SingleSpectrumFit, evaluate_taylor_spectrum
from lwa_catalog.constants import SUBBAND_BANDS_MHZ, SUBBAND_REF_FREQ_MHZ
from lwa_catalog.io import write_table

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coaddR-0.75_subband")
USE_SYNTHETIC = False  # True: inline demo catalog; False: read metacatalog.parquet
WRITE_OUTPUT = False  # True: write metacatalog_spectral.parquet under CATALOG_DIR

layout = CatalogLayout(CATALOG_DIR)
config = SpectralFitConfig(
    bands=SUBBAND_BANDS_MHZ,
    ref_freq_mhz=SUBBAND_REF_FREQ_MHZ,
    flux_kind="total",
)
OUTPUT_PATH = layout.root / "metacatalog_spectral.parquet"

print("CATALOG_DIR =", layout.root.resolve())
print("USE_SYNTHETIC =", USE_SYNTHETIC)
print("WRITE_OUTPUT =", WRITE_OUTPUT)
print("OUTPUT_PATH =", OUTPUT_PATH.resolve())
print("ref_freq_mhz =", config.ref_freq_mhz)

## Load metacatalog

In [ ]:
def _power_law_flux(nu_hz: np.ndarray, *, alpha: float, s_ref: float, nu_ref_hz: float) -> np.ndarray:
    return s_ref * (nu_hz / nu_ref_hz) ** alpha


def _build_synthetic_metacatalog(*, n_sources: int = 60) -> pd.DataFrame:
    """Demo catalog with varied band coverage for diagnostics and SED plots."""
    rng = np.random.default_rng(0)
    bands = SUBBAND_BANDS_MHZ
    nu_hz_all = np.array([float(b.removesuffix("MHz")) * 1e6 for b in bands])
    nu_ref = SUBBAND_REF_FREQ_MHZ * 1e6
    rows: list[dict[str, object]] = []

    for idx in range(n_sources):
        # Coverage tiers: ~1/3 sparse (2 bands), ~1/3 medium (4), ~1/3 well-sampled (8+)
        if idx % 3 == 0:
            use_bands = bands[:2]
        elif idx % 3 == 1:
            use_bands = bands[:4]
        else:
            use_bands = bands[: max(8, 8 + idx % 7)]

        alpha_true = -0.8 + 0.3 * rng.normal()
        s_ref = 10 ** rng.uniform(-1.0, 0.5)
        nu_use = np.array([float(b.removesuffix("MHz")) * 1e6 for b in use_bands])
        flux = _power_law_flux(nu_use, alpha=alpha_true, s_ref=s_ref, nu_ref_hz=nu_ref)

        row: dict[str, object] = {
            "meta_id": idx,
            "RA": float(rng.uniform(0, 360)),
            "DEC": float(rng.uniform(-30, 45)),
            "origin_band": use_bands[0],
            "bands_present": ",".join(use_bands),
        }
        for band, value in zip(use_bands, flux, strict=True):
            row[f"Total_flux_{band}"] = float(value)
            row[f"E_Total_flux_{band}"] = 0.05 * float(value)

        # Adjacent two-point alpha for scatter QA (82/78 MHz pair when both present)
        if "78MHz" in use_bands and "82MHz" in use_bands:
            row["alpha_82_78"] = alpha_true

        rows.append(row)

    return pd.DataFrame(rows)


if USE_SYNTHETIC or not layout.metacatalog().exists():
    if not USE_SYNTHETIC:
        print(f"No metacatalog at {layout.metacatalog()}; using synthetic demo catalog.")
    metacatalog = _build_synthetic_metacatalog()
else:
    metacatalog = read_metacatalog(layout)

print(f"metacatalog rows: {len(metacatalog)}")

## Fit Taylor spectra

In [ ]:
result = fit_metacatalog_spectra(metacatalog, config=config)
fitted = result.metacatalog
print(summarize_spectral_fit(result))

## Diagnostics

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

prefix = config.column_prefix
n_terms_col = f"{prefix}model_n_terms"
n_flux_col = f"{prefix}model_n_flux"
a1_col = f"{prefix}model_a1"

print("Selected model terms (value_counts):")
display(fitted[n_terms_col].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

n_flux = pd.to_numeric(fitted[n_flux_col], errors="coerce")
axes[0].hist(n_flux.dropna(), bins=range(0, int(n_flux.max()) + 2), edgecolor="k", alpha=0.7)
axes[0].set_xlabel("Valid flux channels per source")
axes[0].set_ylabel("Count")
axes[0].set_title("Band coverage")

alpha_cols = [c for c in fitted.columns if c.startswith("alpha_")]
alpha_col = "alpha_82_78" if "alpha_82_78" in fitted.columns else (alpha_cols[0] if alpha_cols else None)
if alpha_col is not None:
    alpha = pd.to_numeric(fitted[alpha_col], errors="coerce")
    a1 = pd.to_numeric(fitted[a1_col], errors="coerce")
    ok = np.isfinite(alpha) & np.isfinite(a1)
    axes[1].scatter(alpha.loc[ok], a1.loc[ok], s=8, alpha=0.5)
    lim = (
        min(alpha.loc[ok].min(), a1.loc[ok].min()),
        max(alpha.loc[ok].max(), a1.loc[ok].max()),
    )
    axes[1].plot(lim, lim, "k--", lw=1, alpha=0.6)
    axes[1].set_xlabel(alpha_col)
    axes[1].set_ylabel(f"{a1_col} (Taylor slope at ν₀)")
    axes[1].set_title(f"Taylor a₁ vs merge-time {alpha_col}")
else:
    axes[1].text(0.5, 0.5, "No alpha_* columns", ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## Example SED overlays

One source each with **≥8**, **4**, and **2** valid flux channels: data points plus best-fit Taylor curve (log–log axes).

In [ ]:
def _pick_example_row(frame: pd.DataFrame, target_n_flux: int) -> pd.Series | None:
    subset = frame.loc[frame[n_flux_col] == target_n_flux]
    if subset.empty:
        subset = frame.loc[frame[n_flux_col] >= target_n_flux]
    if subset.empty:
        return None
    return subset.iloc[0]


def _row_to_fit(row: pd.Series) -> SingleSpectrumFit:
    return SingleSpectrumFit(
        n_terms=int(row[f"{prefix}model_n_terms"]),
        bic=float(row[f"{prefix}model_bic"]),
        chi2_red=float(row[f"{prefix}model_chi2_red"]),
        n_flux=int(row[f"{prefix}model_n_flux"]),
        coeffs=(
            float(row[f"{prefix}model_a0"]),
            float(row[f"{prefix}model_a1"]),
            float(row[f"{prefix}model_a2"]),
            float(row[f"{prefix}model_a3"]),
        ),
        nu0_mhz=float(row[f"{prefix}model_nu0_mhz"]),
    )


examples = [
    ("≥8 bands", _pick_example_row(fitted, 8)),
    ("4 bands", _pick_example_row(fitted, 4)),
    ("2 bands", _pick_example_row(fitted, 2)),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, (title, row) in zip(axes, examples, strict=True):
    if row is None:
        ax.set_title(f"{title} (none found)")
        ax.set_axis_off()
        continue

    nu_hz, flux_jy, err_jy = gather_band_flux_measurements(
        row,
        bands=config.bands,
        flux_kind=config.flux_kind,
    )
    fit = _row_to_fit(row)
    nu_mhz = nu_hz / 1e6
    nu_curve = np.geomspace(nu_mhz.min(), nu_mhz.max(), 100)
    flux_curve = evaluate_taylor_spectrum(nu_curve * 1e6, fit)

    ax.errorbar(nu_mhz, flux_jy, yerr=err_jy, fmt="o", capsize=2, label="data")
    ax.plot(nu_curve, flux_curve, "-", label=f"{fit.n_terms}-term fit")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Frequency (MHz)")
    ax.set_title(f"{title} (meta_id={row.get('meta_id', '?')})")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Total flux (Jy)")
plt.tight_layout()
plt.show()

## Optional write

Writes `metacatalog_spectral.parquet` (original columns + `spec_*` fit columns). Set `WRITE_OUTPUT = True` in the config cell.

In [ ]:
if WRITE_OUTPUT:
    write_table(result.metacatalog, OUTPUT_PATH, include_extras=True)
    print(f"Wrote {OUTPUT_PATH}")
else:
    print("Skipping write (WRITE_OUTPUT=False)")